# Answers: 6 · Smarter strategies

Worked answers to the **Try it yourself** exercises at the end of
[`6_smarter_strategies.ipynb`](../6_smarter_strategies.ipynb). Try each one yourself first. If you're stuck, run the cell here and
compare. Each answer runs on the same real data as the lesson, so you'll see real results,
not made-up ones.

The first code cell recreates just what the exercises need from the lesson, so this notebook
runs on its own.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from tour import *
from adaptive_acq.campaign import Campaign
from adaptive_acq.controllers import Controller, FixedBeta, Schedule, BatchSplitThompson
from adaptive_acq.sim_env import BootstrapRidge

scores, embeddings = load_mini()
y = scores["score"].to_numpy(); N = len(y)
true_top = np.zeros(N, dtype=bool); true_top[np.argsort(y)[:100]] = True
X = embeddings["unimol2"].astype(np.float64); X = (X - X.mean(0)) / (X.std(0) + 1e-6)
U, S, _ = np.linalg.svd(X - X.mean(0), full_matrices=False); X64 = U[:, :64] * S[:64]

def make_campaign(seed, init=105, batch=35, rounds=9, prefilter_mult=0.0):
    surrogate = BootstrapRidge(M=5, alpha=10.0, seed=seed)
    start = np.random.default_rng(seed).choice(N, init, replace=False)
    return Campaign(n_pool=N, fit_predict=lambda li, yhb, ci: surrogate(X64, li, yhb, ci),
                    score_fn=lambda idx: y[idx], init_idx=start, batch_size=batch, n_rounds=rounds,
                    k_obs=100, true_top_mask=true_top, prefilter_mult=prefilter_mult)

class ExploreEarlyOnly(Controller):
    name, arms = "early", (0.0, 0.5)
    def plan(self, ctx, rnd, n_rounds, batch_size):
        return [(0, 0.5 if rnd < 3 else 0.0, batch_size)]

def mean_recall(make_controller, **campaign_kw):
    return np.mean([make_campaign(s, **campaign_kw).run(make_controller(s))[-1]["topk_recall"] for s in range(5)])
print("ready")

---
## Exercise 1: explore only in the **last** 3 rounds

In [ ]:
class ExploreLateOnly(Controller):
    name, arms = "late", (0.0, 0.5)
    def plan(self, ctx, rnd, n_rounds, batch_size):
        return [(0, 0.5 if rnd >= n_rounds - 3 else 0.0, batch_size)]

for name, make in [("greedy", lambda s: FixedBeta(0.0)), ("explore early", lambda s: ExploreEarlyOnly()),
                   ("explore late", lambda s: ExploreLateOnly())]:
    print(f"{name:14s} mean final recall (5 seeds): {mean_recall(make):.3f}")

**Answer.** About the same as exploring early, and both are worse than plain greedy: greedy 0.612, explore early 0.584, explore late 0.580 (5 seeds each). Exploring *late* has an extra drawback in principle: whatever it learns from its unusual picks, there are no rounds left to use it. Here it makes little difference which end you explore at. Every round spent exploring is a round not spent docking the best guesses, and with a decent model that costs more than it gains.

---
## Exercise 2: a generous budget (start 210, then 5 × 210 = 1,260 molecules)

In [ ]:
contenders = {
    "greedy (beta 0)":      lambda s: FixedBeta(0.0),
    "beta 0.3":             lambda s: FixedBeta(0.3),
    "beta 1.0":             lambda s: FixedBeta(1.0),
    "explore, then exploit": lambda s: Schedule("explore_then_exploit", beta_hi=0.5, beta_lo=0.0),
    "Thompson bandit":      lambda s: BatchSplitThompson(seed=s),
}
rows = {name: {"tight budget (420)": mean_recall(make),
               "generous budget (1,260)": mean_recall(make, init=210, batch=210, rounds=5)}
        for name, make in contenders.items()}
pd.DataFrame(rows).T.round(3)

**Answer.** With the generous budget all strategies score much higher (0.77–0.94), and the gaps between the sensible ones shrink. Greedy (0.934) and the Thompson bandit (0.938) are tied, with β = 0.3 (0.918) and explore-then-exploit (0.922) just behind. Only heavy exploration (β = 1.0: 0.766) still clearly hurts. With more budget the model gets good enough that nearly any reasonable rule finds most of the best molecules. That's the ceiling effect from notebook 2 again, and another reason the project tested on tight budgets, where differences can show.

---
## Exercise 3: how small can the prefilter's shortlist get?

The shortlist holds `prefilter_mult × budget` molecules (budget = 420 here). Below 1, it's
smaller than the number of molecules we want to dock.

In [ ]:
import warnings
for mult in [0, 3, 1.5, 1.0, 0.5]:
    label = f"prefilter_mult {mult:<4}  shortlist {int(mult * 420) if mult else 'none':>5}"
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")          # hide numpy's "mean of empty slice" warnings
            hs = [make_campaign(s, prefilter_mult=mult).run(FixedBeta(0.0)) for s in range(5)]
        print(f"{label}   recall {np.mean([h[-1]['topk_recall'] for h in hs]):.3f}")
    except Exception as err:
        print(f"{label}   CRASHED: {type(err).__name__}: {err}")

**Answer.** Shrinking the shortlist costs recall well before it breaks. Three times the budget (1,260 molecules) is as good as no prefilter (0.616 vs 0.612), 1.5× drops to 0.582, and 1× (exactly the 420 we plan to dock) drops to 0.514. A tight shortlist leaves no room to correct round 1's mistakes: anything the first model ranked badly can never be picked later.

Below 1× it **crashes**. With `prefilter_mult=0.5` the shortlist holds 210 molecules, fewer than we want to dock. After 6 rounds every shortlisted molecule is docked, no candidates are left, and `featurize` in `adaptive_acq/state.py` divides by the size of an empty set: `ZeroDivisionError: division by zero`. That's a genuine **edge case** the project's code doesn't guard against, the kind of thing a test (notebook 8) would catch. The project only ever uses multipliers like 10×, so it never hits it.